# Backtest-vs-Live Audit Dashboard

Phase 0 of the *Path A* execution plan. This notebook wires together:
- `log_parser`  — parse a QC algorithm log into structured trade records
- `qc_api`      — fetch QC backtest orders for the same period
- `compare`     — produce a gap-attribution report
- `harsh_simulator` — show what the backtest would look like under live-realistic assumptions
- `regime_runner`   — rank parameter sets by walk-forward survivability
- `report`      — render any of the above as standalone HTML

The default cells below produce the MG36 audit. Edit the project / backtest / log paths to point at your own.

## 1. Parse the live log

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../..'))

from backtest_audit.log_parser import parse_log_file, pair_trades

LOG = os.path.abspath('../fixtures/mg36_paper_2026-03-16.txt')
parsed = parse_log_file(LOG)
summary = parsed.summary()
for k, v in summary.items():
    print(f'  {k:>22}: {v}')

In [ ]:
trades = pair_trades(parsed)
for t in trades:
    print(f'{t.symbol:>10}  entry={t.entry_price:.5f}  exit={t.exit_price:.5f}  '
          f'gross={t.gross_pct*100:+.3f}%  held={t.held_seconds:.0f}s  '
          f'score={t.score}  e_slip={t.entry_slippage_bps}bp  x_slip={t.exit_slippage_bps}bp')

## 2. Build the gap report (live-only or live-vs-backtest)

In [ ]:
from backtest_audit.compare import build_report, report_from_paired_log

# Live-only (no backtest counterpart yet)
rep = report_from_paired_log(parsed)
print(rep.summary_text())

## 3. (Optional) Compare against a QC backtest of the same period

Set the `QC_USER_ID` and `QC_API_TOKEN` env vars before running. Identify the `project_id` and `backtest_id` of the backtest you want to compare against.

In [ ]:
from backtest_audit.qc_api import QCClient

if 'QC_USER_ID' in os.environ or 'QC_UID' in os.environ:
    qc = QCClient.from_env()
    print('QC auth:', qc.authenticate())
    # Edit these two lines to point at the project + backtest:
    PROJECT_ID  = 29075772        # Machine Gun v36
    BACKTEST_ID = None            # set this to a backtest of the same date range
    if BACKTEST_ID:
        bt = qc.read_backtest(PROJECT_ID, BACKTEST_ID)
        print('backtest:', bt.get('name'), bt.get('created'))
        # TODO Phase 0 follow-up: convert QC orders → CompletedTrade list
        # and feed into build_report()
    else:
        print('Set BACKTEST_ID to compare against a backtest of the same period.')
else:
    print('QC_USER_ID / QC_API_TOKEN not set — skipping QC fetch')

## 4. Show what the harsh simulator would have predicted

Running 1000 simulated KASUSD-style trades to verify the harsh sim slippage envelope matches what we observed live.

In [ ]:
import statistics
from backtest_audit.harsh_simulator import HarshConfig, HarshFillSimulator

sim = HarshFillSimulator(HarshConfig(seed=2026, reject_rate_normal=0.0))
for tier in ('major', 'large', 'mid', 'micro'):
    rt = []
    for _ in range(2000):
        b = sim.simulate_fill('X', 'Buy',  True, 100, 100, tier=tier)
        s = sim.simulate_fill('X', 'Sell', True, 100, 100, tier=tier)
        rt.append(b.slippage_bps + s.slippage_bps)
    print(f'  tier={tier:<6}  round-trip slip:  '
          f'mean={statistics.mean(rt):6.1f}bp  p50={sorted(rt)[1000]:6.1f}bp  p99={sorted(rt)[1980]:6.1f}bp')

## 5. Run the universe gate against your current candidate list

In [ ]:
from Pulse.universe import UniverseGate, SymbolStats, SymbolTierClassifier, screen

# Replace this with your real per-symbol stats (from QC history calls)
candidates = [
    SymbolStats('BTCUSD',  40_000_000_000, 1.0,  50000, 2000),
    SymbolStats('SOLUSD',   2_000_000_000, 8.0,    170, 1000),
    SymbolStats('INJUSD',      80_000_000, 15.0,    20, 600),
    SymbolStats('KASUSD',      15_000_000, 25.0,  0.04, 300),
    SymbolStats('FARTCOINUSD',    300_000, 120.0, 0.21,  15),  # bad — should reject
    SymbolStats('PEAQUSD',        400_000,  80.0, 0.08,  20),  # bad
]
out = screen(candidates)
print(f'Eligible: {[s.symbol for s in out["eligible"]]}')
print(f'Rejected:')
for d in out['rejected']:
    print(f'  {d.symbol:<14} reasons={list(d.reasons)}')
print(f'\nTier limits:')
for sym, lim in out['tiers'].items():
    print(f'  {sym:<10} tier={lim["tier"]:<6} max_pos=${lim["max_pos_usd"]:>6.0f}  slip_budget={lim["slip_budget_bps"]:.0f}bp')

## 6. Render full HTML audit page

In [ ]:
from backtest_audit.report import write_audit_page

out_path = write_audit_page(
    '../reports/mg36_audit.html',
    title='MG36 Phase 0 Audit',
    parsed_log=parsed,
    gap_report=rep,
    notes='Re-run from notebook on ' + str(__import__('datetime').datetime.now()),
)
print(f'Wrote: {out_path}')